## Face-to-BMI

Modeling

## 1.Pacakges

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import random
from PIL import Image
from collections import Counter
import cv2
from deepface import DeepFace
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Flatten,  Dense, Dropout, BatchNormalization
#from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.legacy import Adam  # for M1/M2 Mac
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## 2. Read clean data

In [3]:
# Get working path
working_path = Path.cwd()
print(working_path)
# Verify that the Data folder exists
data_path = working_path / 'Data'
if data_path.is_dir():
    print("The data folder exists:",data_path )
else:
    print('Please download the data Face-to-BMI: Using Computer Vision to Infer Body Mass Index on Social Media Paper')

/Users/mattl/Documents/U Chicago/Q3/ML_2/Final Project/BMI
The data folder exists: /Users/mattl/Documents/U Chicago/Q3/ML_2/Final Project/BMI/Data


In [4]:
# Read label data
df_modeling = pd.read_parquet('Data/df_EDA_clean.parquet', engine='pyarrow')
print('Shape:', df_modeling.shape)
df_modeling.info()

Shape: (3962, 15)
<class 'pandas.DataFrame'>
RangeIndex: 3962 entries, 0 to 3961
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   image_index        3962 non-null   int64   
 1   bmi                3962 non-null   float64 
 2   gender             3962 non-null   str     
 3   is_training        3962 non-null   int64   
 4   name               3962 non-null   str     
 5   BMI_category       3962 non-null   category
 6   index_pair         3962 non-null   int64   
 7   flag_before_after  3962 non-null   str     
 8   images_path        3962 non-null   str     
 9   brightness         3962 non-null   float64 
 10  contrast           3962 non-null   float64 
 11  blur               3962 non-null   float64 
 12  flag_brightness    3962 non-null   int64   
 13  flag_contrast      3962 non-null   int64   
 14  flag_blur          3962 non-null   int64   
dtypes: category(1), float64(4), int64(6), str(4)
mem

In [5]:
# Number classes
num_classes = ['under-weight', 'normal', 'overweight', 'moderately obese', 'severely obese', 'very severely obese']
mapping = {'under-weight': 0, 'normal': 1, 'overweight': 2,'moderately obese': 3,'severely obese':4, 'very severely obese': 5 }
df_modeling['BMI_category_encode'] = df_modeling['BMI_category'].map(mapping)

# Split data into train and test
df_train_raw = df_modeling[df_modeling['is_training']==1]
df_test = df_modeling[df_modeling['is_training']==0]

In [6]:
display(df_modeling['BMI_category'].value_counts())
display(df_modeling['BMI_category_encode'].value_counts())

BMI_category
overweight             1087
moderately obese        885
very severely obese     695
normal                  663
severely obese          625
under-weight              7
Name: count, dtype: int64

BMI_category_encode
2    1087
3     885
5     695
1     663
4     625
0       7
Name: count, dtype: int64

In [7]:
# Establish train and validation datasets to create holdout set of test
initial_rows  = df_train_raw.shape[0]
n_train = (df_train_raw.shape[0]) * 0.8
n_train = int(n_train)
print('Intial rows {}, it will be divide into training: {} and in validation {}'.format(initial_rows, n_train,initial_rows-n_train) )
# Split train and validation base on the index
# Train
df_train = df_train_raw[0:n_train]
y_train = df_train['BMI_category_encode'].to_numpy()
# Validation
df_val = df_train_raw[n_train: initial_rows]
y_val = df_val['BMI_category_encode'].to_numpy()
# Test
y_test =  df_test['BMI_category_encode'].to_numpy()

print('\nSanity check:')
print('In training shape features: {} and shape classes: {}  '.format(df_train.shape, y_train.shape ))
print('In validation shape features: {} and shape classes: {}  '.format(df_val.shape, y_val.shape ))

Intial rows 3210, it will be divide into training: 2568 and in validation 642

Sanity check:
In training shape features: (2568, 16) and shape classes: (2568,)  
In validation shape features: (642, 16) and shape classes: (642,)  


In [8]:
# Check distribution
print('train')
display(df_train[['bmi']].describe())
display(df_train[['gender']].value_counts(normalize=True))

print('val')
display(df_val[['bmi']].describe())
display(df_val[['gender']].value_counts(normalize=True))

print('test')
display(df_test[['bmi']].describe())
display(df_test[['gender']].value_counts(normalize=True))

train


,bmi
count,2568.000000
mean,32.376457
std,8.024783
min,17.716216
25%,26.258366
50%,31.001126
75%,36.956668
max,81.211930


gender
Male      0.610981
Female    0.389019
Name: proportion, dtype: float64

val


,bmi
count,642.000000
mean,32.666025
std,7.988843
min,19.483497
25%,26.561999
50%,30.897279
75%,37.842392
max,58.571712


gender
Male      0.557632
Female    0.442368
Name: proportion, dtype: float64

test


,bmi
count,752.000000
mean,33.698354
std,9.219863
min,18.651020
25%,26.872880
50%,31.947427
75%,38.867300
max,85.987061


gender
Male      0.567819
Female    0.432181
Name: proportion, dtype: float64

The distribution between the train and test used by the author in the paper is very similar, the test has a slightly higher average BMI compared to the train, but the quantiles of the BMI are very similar between the two datasets, which will not be an issue in the modeling step. In addition, there is more concentration of males in the train dataset with 60%, in contrast with the test 56%, it is something we need to consider in the model train process.

## 3. Modeling

## 3.1 Modeling aproach 1: Baseline

In [9]:
# VGGFace mean pixel values (NOT ImageNet means)
VGGFACE_MEAN = [93.5940, 104.7624, 129.1863]  # BGR order
IMG_SIZE = (224, 224)

def preprocess_vggface(img_path):
    """
    Load and preprocess a face image for VGGFace.
    - Resize to 224x224
    - Convert to BGR (OpenCV default)
    - Subtract VGGFace mean (no /255 normalization)
    """
    # Load image
    img = cv2.imread(str(img_path))
    
    if img is None:
        raise ValueError(f"Could not load image: {img_path}")
    
    # Resize
    img = cv2.resize(img, IMG_SIZE)
    
    # Convert to float32
    img = img.astype(np.float32)
    
    # Subtract VGGFace channel means (BGR order)
    img[..., 0] -= VGGFACE_MEAN[0]  # B
    img[..., 1] -= VGGFACE_MEAN[1]  # G
    img[..., 2] -= VGGFACE_MEAN[2]  # R
    
    return img  # shape: (224, 224, 3)

In [11]:
# Prprecoses  images
X_train_list = []
X_val_list = []
X_test_list = []

# Training preprocess
for path_img in df_train.loc[:,'images_path']:
    X_train_list.append(preprocess_vggface(path_img))\

# Val preprocess
for path_img in df_val.loc[:,'images_path']:
    X_val_list.append(preprocess_vggface(path_img))

# Test preprocess
for path_img in df_test.loc[:,'images_path']:
    X_test_list.append(preprocess_vggface(path_img))

# Transform to numpy array
X_train = np.array(X_train_list, dtype=np.float32)
X_val = np.array(X_val_list, dtype=np.float32)
X_test = np.array(X_test_list, dtype=np.float32)

# Change type of the classs
y_train = y_train.astype(np.float32)
y_val   = y_val.astype(np.float32)
y_test  = y_test.astype(np.float32)

print('Features:')
print(f"Train : {X_train.shape}")  
print(f"Val   : {X_val.shape}")    
print(f"Test  : {X_test.shape}")
    
print('Classes:')
print(f"Train : {y_train.shape}")  
print(f"Val   : {y_val.shape}")    
print(f"Test  : {y_test.shape}")    


Features:
Train : (2568, 224, 224, 3)
Val   : (642, 224, 224, 3)
Test  : (752, 224, 224, 3)
Classes:
Train : (2568,)
Val   : (642,)
Test  : (752,)


Initial architecture

In [12]:
model_vgg   = DeepFace.build_model("VGG-Face")
base = model_vgg.model
# Check layer of the model
for i, layer in enumerate(base.layers):
    print(f"{i:3d} | {layer.name:35s} | {str(layer.output_shape):25s} | trainable={layer.trainable}")

  0 | zero_padding2d_input                | [(None, 224, 224, 3)]     | trainable=True
  1 | zero_padding2d                      | (None, 226, 226, 3)       | trainable=True
  2 | conv2d                              | (None, 224, 224, 64)      | trainable=True
  3 | zero_padding2d_1                    | (None, 226, 226, 64)      | trainable=True
  4 | conv2d_1                            | (None, 224, 224, 64)      | trainable=True
  5 | max_pooling2d                       | (None, 112, 112, 64)      | trainable=True
  6 | zero_padding2d_2                    | (None, 114, 114, 64)      | trainable=True
  7 | conv2d_2                            | (None, 112, 112, 128)     | trainable=True
  8 | zero_padding2d_3                    | (None, 114, 114, 128)     | trainable=True
  9 | conv2d_3                            | (None, 112, 112, 128)     | trainable=True
 10 | max_pooling2d_1                     | (None, 56, 56, 128)       | trainable=True
 11 | zero_padding2d_4                    |

In [13]:
# Step initial everything to False
base.trainable = False

# Unfreeze only Block 5 (last conv block → max_pooling2d_4)
unfreeze_from = "zero_padding2d_10"  # start of block 5
freeze = True
for layer in base.layers:
    if layer.name == unfreeze_from:
        freeze = False
    layer.trainable = not freeze
# Check number of trainable and freeze layers
print("Freeze layers:", sum(1 for l in base.layers if l.trainable==False))
print("Trainable layers:", sum(1 for l in base.layers if l.trainable))

Freeze layers: 25
Trainable layers: 11


 Build the Fine-Tuned Model

In [14]:
# Cut at last pooling layer
cut = base.get_layer("max_pooling2d_4").output
x   = GlobalAveragePooling2D()(cut)

# Custom head for BMI regression (4 extra layers)
# Block 1 
x = Dense(512, activation='relu')(x)  # addtional layer 1
x = BatchNormalization()(x) # addtional layer 2
x = Dropout(0.5)(x)
# Block 2
x = Dense(256, activation='relu')(x) # addtional layer 3
x = BatchNormalization()(x) # addtional layer 4
x = Dropout(0.3)(x)
output = Dense(1, activation='linear', name='bmi_output')(x) 


model_tune = Model(inputs=base.input, outputs=output)
print("Freeze layers:", sum(1 for l in model_tune.layers if l.trainable==False))
print("Trainable layers:", sum(1 for l in model_tune.layers if l.trainable))

Freeze layers: 25
Trainable layers: 15


In [15]:
# Extract the addtional layers add 
last_4_layers = model_tune.layers[-8:]

# 3. Print a custom summary for these layers
print(f"{'Layer (type)':<60} {'Output Shape':<25} {'Param #'}")
print("=" * 120)

total_params = 0

for layer in last_4_layers:
    # Format layer name and class type
    layer_info = f"{layer.name} ({layer.__class__.__name__})"
    
    # Handle multiple output shapes (if applicable)
    if isinstance(layer.output_shape, list):
        output_shape = str(layer.output_shape[0])
    else:
        output_shape = str(layer.output_shape)
        
    # Get parameter count
    params = layer.count_params()
    total_params += params
    
    print(f"{layer_info:<60} {output_shape:<25} {params}")

print("=" * 120)
print(f"Total params addtional layers add to be tune step: {total_params:,}")

Layer (type)                                                 Output Shape              Param #
global_average_pooling2d (GlobalAveragePooling2D)            (None, 512)               0
dense (Dense)                                                (None, 512)               262656
batch_normalization (BatchNormalization)                     (None, 512)               2048
dropout_2 (Dropout)                                          (None, 512)               0
dense_1 (Dense)                                              (None, 256)               131328
batch_normalization_1 (BatchNormalization)                   (None, 256)               1024
dropout_3 (Dropout)                                          (None, 256)               0
bmi_output (Dense)                                           (None, 1)                 257
Total params addtional layers add to be tune step: 397,313


In [ ]:
# ─── PHASE 1: Train head only (frozen base) ──────────────────────
base.trainable = False

model_tune.compile(
    optimizer=Adam(learning_rate=1e-3),  # higher LR — only head trains
    loss='huber',
    metrics=['mae']
)

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_phase1.keras", save_best_only=True, verbose=1)
]

print("=== PHASE 1: Training head only ===")
history1 = model_tune.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks
)


# ─── PHASE 2: Unfreeze & fine-tune top layers (lower LR) ──────────
freeze = True
for layer in base.layers:
    if layer.name == "zero_padding2d_10":  # unfreeze from block 5
        freeze = False
    layer.trainable = not freeze

model_tune.compile(
    optimizer=Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss='huber',
    metrics=['mae']
)

callbacks2 = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-8, verbose=1),
    ModelCheckpoint("best_phase2.keras", save_best_only=True, verbose=1)
]

print("=== PHASE 2: Fine-tuning top layers ===")
history2 = model_tune.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,          # smaller batch for fine-tuning
    callbacks=callbacks2
)

=== PHASE 1: Training head only ===
Epoch 1/30
